# 06. 메타데이터, 태그 및 필터링

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `config` 파라미터로 `run_name`, `tags`, `metadata`를 Trace에 추가할 수 있어요
2. `tracing_context`를 사용해 코드 블록 내 모든 Trace에 공통 메타데이터를 설정할 수 있어요
3. `get_current_run_tree()`로 실행 중인 Trace에 동적 메타데이터를 추가할 수 있어요
4. LangSmith UI에서 태그, 메타데이터, 상태 기반 필터링을 사용해 원하는 Trace를 찾을 수 있어요

## 사전 지식

- `05-Memory-Tracing` 노트북 완료
- `@traceable` 데코레이터 기본 사용법
- LCEL 체인 구성 경험

## 개념 설명: 메타데이터와 태그의 역할

LangSmith에서 수백, 수천 개의 Trace를 효과적으로 관리하려면 **메타데이터**와 **태그**를 활용해야 해요.

### 메타데이터 vs 태그

| 항목 | 메타데이터(metadata) | 태그(tags) |
|------|---------------------|------------|
| 형식 | 키-값 쌍 딕셔너리 | 문자열 리스트 |
| 용도 | 구체적 정보 저장 | 카테고리 분류 |
| 예시 | `{"user_id": "u123", "version": "v2"}` | `["production", "rag", "v2"]` |
| 필터링 | 키-값 조합으로 정확 검색 | 태그명으로 그룹 검색 |

### 메타데이터 설정 방법 3가지

```mermaid
flowchart LR
    A[메타데이터 설정 방법] --> B[config 파라미터]
    A --> C[tracing_context 컨텍스트 관리자]
    A --> D[get_current_run_tree 동적 추가]

    B --> B1["chain.invoke(..., config={...})<br/>단일 호출에 적용"]
    C --> C1["with ls.tracing_context(...)<br/>블록 내 모든 호출에 적용"]
    D --> D1["run.metadata['key'] = value<br/>실행 중 동적으로 추가"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C,D process
    class B1,C1,D1 output
```

> 🔑 **핵심 개념**: `tracing_context`는 새로운 Span(Trace)을 **생성하지 않아요**. 기존 체인 호출의 기본값(default)을 설정하는 역할만 해요.

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경변수 로드 및 프로젝트 설정
# ---------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()

# 이 노트북 전용 LangSmith 프로젝트
os.environ["LANGSMITH_PROJECT"] = "Ch06-Metadata-Tags"

print("프로젝트:", os.environ["LANGSMITH_PROJECT"])

프로젝트: Ch06-Metadata-Tags


In [2]:
# ---------------------------------------------------
# 기본 체인 설정 (이 노트북 전체에서 재사용)
# ---------------------------------------------------
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_template("{topic}에 대해 1문장으로 설명해 주세요.")
parser = StrOutputParser()

# 기본 체인
chain = prompt | model | parser

print("기본 체인 준비 완료!")

기본 체인 준비 완료!


## 1. config 파라미터로 메타데이터 추가

가장 기본적인 방법이에요. 체인 호출 시 `config` 파라미터에 `run_name`, `tags`, `metadata`를 지정해요.

> 🎯 **강의 포인트**: LangSmith UI에서 `run_name`을 설정하면 Trace 목록에서 의미 있는 이름으로 표시돼요. `ChatOpenAI`, `RunnableSequence` 같은 기본 이름 대신 비즈니스 의미를 담은 이름을 사용하세요.

In [3]:
# ---------------------------------------------------
# config로 run_name, tags, metadata 설정
# ---------------------------------------------------
# 단일 호출에 메타데이터를 추가하는 방법이에요

result = chain.invoke(
    {"topic": "트랜스포머"},
    config={
        "run_name": "transformer-explanation",  # Trace 이름 (UI에 표시됨)
        "tags": ["education", "nlp", "v1"],     # 카테고리 분류용 태그
        "metadata": {
            "user_id": "student_001",            # 사용자 식별
            "session_type": "study",             # 세션 유형
            "topic_category": "deep_learning",   # 주제 카테고리
            "environment": "development"         # 환경 정보
        }
    }
)

print(f"결과: {result}")
print("\nLangSmith에서 'transformer-explanation' Trace를 찾아서 메타데이터를 확인해 보세요!")

결과: 트랜스포머는 자연어 처리에서 주로 사용되는 딥러닝 모델로, 입력 데이터의 모든 단어 간의 관계를 동시에 고려하여 효율적으로 정보를 처리하는 구조를 가지고 있습니다.

LangSmith에서 'transformer-explanation' Trace를 찾아서 메타데이터를 확인해 보세요!


In [4]:
# ---------------------------------------------------
# 여러 Trace에 다른 태그/메타데이터 적용
# ---------------------------------------------------
# 태그를 활용해 다양한 Trace를 분류해요

topics = [
    ("BERT", ["nlp", "bert"], "nlp_model"),
    ("GPT", ["nlp", "gpt", "generative"], "nlp_model"),
    ("FAISS", ["vector_db", "search"], "database"),
    ("LangChain", ["framework", "langchain"], "tool"),
]

for topic, tags, category in topics:
    result = chain.invoke(
        {"topic": topic},
        config={
            "run_name": f"explain-{topic.lower()}",
            "tags": tags,
            "metadata": {"category": category}
        }
    )
    print(f"[{topic}] ({', '.join(tags)})")
    print(f"  → {result[:100]}")

print("\nLangSmith에서 'nlp' 태그로 필터링해서 관련 Trace만 모아 보세요!")

[BERT] (nlp, bert)
  → BERT(Bidirectional Encoder Representations from Transformers)는 문맥을 양방향으로 이해하여 자연어 처리 작업을 수행하는 데 사용되는
[GPT] (nlp, gpt, generative)
  → GPT는 자연어 처리를 기반으로 한 인공지능 모델로, 인간과 유사한 방식으로 텍스트를 생성하고 이해하는 능력을 갖추고 있습니다.
[FAISS] (vector_db, search)
  → FAISS(Facebook AI Similarity Search)는 대규모 벡터 데이터셋에서 효율적으로 유사성을 검색할 수 있도록 설계된 라이브러리입니다.
[LangChain] (framework, langchain)
  → LangChain은 다양한 언어 모델과 데이터 소스를 통합하여 자연어 처리 애플리케이션을 구축하고 관리할 수 있도록 돕는 프레임워크입니다.

LangSmith에서 'nlp' 태그로 필터링해서 관련 Trace만 모아 보세요!


## 2. @traceable에서 tags와 metadata 설정

`@traceable` 데코레이터에서도 `tags`와 `metadata`를 설정할 수 있어요.

> 💡 **실무 팁**: `@traceable`의 메타데이터는 **정적(static)** 정보를 설정할 때 좋아요. 호출마다 달라지는 동적 정보는 `get_current_run_tree()`를 사용하세요.

In [5]:
# ---------------------------------------------------
# @traceable에서 정적 메타데이터 설정
# ---------------------------------------------------
# 함수 정의 시점에 고정된 메타데이터를 설정해요
from langsmith import traceable, get_current_run_tree

@traceable(
    name="rag-retriever",                       # Trace 이름
    run_type="retriever",                       # Retriever 타입으로 표시돼요
    tags=["rag", "vector-search"],              # 정적 태그
    metadata={"component": "retriever", "version": "1.0"}  # 정적 메타데이터
)
def mock_retriever(query: str) -> list[dict]:
    """RAG 리트리버 모의 함수예요 (실제로는 벡터 DB 검색)"""
    # 실제 검색 대신 모의 데이터 반환
    return [
        {"content": f"'{query}'에 관한 첫 번째 문서", "score": 0.95},
        {"content": f"'{query}'에 관한 두 번째 문서", "score": 0.87}
    ]

@traceable(
    name="rag-generator",
    run_type="chain",
    tags=["rag", "generation"]
)
def mock_generator(query: str, docs: list[dict]) -> str:
    """RAG 생성기 모의 함수예요"""
    context = "\n".join([d["content"] for d in docs])
    prompt_text = f"컨텍스트: {context}\n\n질문: {query}"
    return chain.invoke({"topic": prompt_text})

# 실행
query = "임베딩이란 무엇인가요?"
docs = mock_retriever(query)
answer = mock_generator(query, docs)

print(f"질문: {query}")
print(f"검색된 문서 수: {len(docs)}")
print(f"답변: {answer}")

질문: 임베딩이란 무엇인가요?
검색된 문서 수: 2
답변: 임베딩이란 고차원 데이터를 저차원 공간에 매핑하여 의미 있는 표현을 생성하는 방법으로, 주로 자연어 처리와 기계 학습에서 사용됩니다.


## 3. get_current_run_tree()로 동적 메타데이터 추가

함수 실행 중에 결과에 따라 동적으로 메타데이터를 추가할 수 있어요.

> 🔑 **핵심 개념**: `get_current_run_tree()`는 현재 실행 중인 Trace 객체를 반환해요. 이 객체를 통해 처리 결과에 따른 메타데이터를 추가하거나 태그를 업데이트할 수 있어요.

In [6]:
# ---------------------------------------------------
# get_current_run_tree()로 동적 메타데이터 추가
# ---------------------------------------------------
# 실행 결과에 따라 동적으로 메타데이터를 설정해요
import time

@traceable(name="quality-aware-chat")
def quality_aware_chat(question: str) -> str:
    """응답 품질 메타데이터를 동적으로 추가하는 함수예요"""
    start_time = time.time()

    # LLM 호출
    response = chain.invoke({"topic": question})

    # 처리 시간 측정
    processing_time = time.time() - start_time

    # 응답 품질 판단 (간단한 휴리스틱)
    word_count = len(response.split())
    quality = "high" if word_count > 20 else "low"

    # 현재 실행 중인 Trace에 동적 메타데이터 추가
    run = get_current_run_tree()
    if run:
        # 처리 결과에 따른 메타데이터 추가
        run.metadata["processing_time_sec"] = round(processing_time, 3)
        run.metadata["response_word_count"] = word_count
        run.metadata["response_quality"] = quality

        # 품질에 따라 태그도 동적으로 추가
        run.tags.extend([f"quality:{quality}", f"words:{word_count // 10 * 10}"])

    return response

# 여러 질문으로 테스트
questions = [
    "AI란?",
    "딥러닝이 다른 머신러닝 방법과 다른 점은 무엇인가요?"
]

for q in questions:
    result = quality_aware_chat(q)
    print(f"질문: {q[:50]}")
    print(f"응답: {result[:100]}...\n")

질문: AI란?
응답: AI(인공지능)는 인간의 지능을 모방하여 학습, 문제 해결, 의사결정 등을 수행할 수 있는 컴퓨터 시스템이나 소프트웨어를 의미합니다....

질문: 딥러닝이 다른 머신러닝 방법과 다른 점은 무엇인가요?
응답: 딥러닝은 다층 신경망을 사용하여 데이터의 복잡한 패턴을 자동으로 학습하는 반면, 전통적인 머신러닝 방법은 주로 수작업으로 특징을 추출하고 단순한 모델을 사용하는 점에서 차이가 있습...



## 4. tracing_context로 범위 설정

`tracing_context`는 코드 블록 내 모든 Trace에 공통 메타데이터를 적용해요. 새 Span을 만들지 않고 기본값만 설정해요.

> ⚠️ **자주 하는 실수**: `tracing_context`가 새로운 Span을 만들 것이라고 오해하는 경우가 있어요. 실제로는 **Span 없이** 기본값만 설정해요. LangSmith Trace에서 `tracing_context` 이름의 Span은 보이지 않아요.

In [8]:
# ---------------------------------------------------
# tracing_context: 블록 내 모든 Trace에 공통 메타데이터
# ---------------------------------------------------
# with 블록 안의 모든 체인 호출에 동일한 메타데이터가 적용돼요
import langsmith as ls

# 사용자 A의 요청 처리 (개발 환경)
print("=== 사용자 A 요청 처리 (개발) ===")
with ls.tracing_context(
    project_name="LangSmith-DEV",          # 별도 프로젝트로 라우팅
    tags=["user:A", "environment:dev"],
    metadata={
        "user_id": "user_A",
        "environment": "development",
        "request_batch": "batch_001"
    }
):
    # 이 블록 안의 모든 체인 호출이 위 메타데이터를 가져요
    r1 = chain.invoke({"topic": "Transformer"})
    r2 = chain.invoke({"topic": "Attention"})
    print(f"Transformer: {r1[:80]}")
    print(f"Attention: {r2[:80]}")

print("\n=== 사용자 B 요청 처리 (운영) ===")
with ls.tracing_context(
    project_name="LangSmith-PROD",         # 운영 환경은 다른 프로젝트
    tags=["user:B", "environment:prod"],
    metadata={
        "user_id": "user_B",
        "environment": "production",
        "request_batch": "batch_002"
    }
):
    r3 = chain.invoke({"topic": "BERT"})
    print(f"BERT: {r3[:80]}")

print("\nLangSmith에서 두 프로젝트(DEV/PROD)에 각각 Trace가 기록됐어요!")

=== 사용자 A 요청 처리 (개발) ===
Transformer: Transformer는 자연어 처리 및 기타 시퀀스 데이터 작업에서 사용되는 딥러닝 모델로, 자기 주의(attention) 메커니즘을 통해 입력
Attention: Attention은 입력 데이터의 특정 부분에 집중하여 중요한 정보를 강조하고, 이를 통해 모델의 성능을 향상시키는 기법입니다.

=== 사용자 B 요청 처리 (운영) ===
BERT: BERT(Bidirectional Encoder Representations from Transformers)는 문맥을 양방향으로 이해하여 자연

LangSmith에서 두 프로젝트(DEV/PROD)에 각각 Trace가 기록됐어요!


## 5. 프로젝트 관리: dev/staging/prod 분리

실무에서는 환경별로 별도 LangSmith 프로젝트를 사용해요. `LANGCHAIN_PROJECT` 환경변수 또는 `tracing_context`로 동적 라우팅이 가능해요.

> 💡 **실무 팁**: 환경별 프로젝트 분리를 통해 운영 데이터와 개발/테스트 데이터를 명확히 구분할 수 있어요. 비용도 환경별로 집계할 수 있어요.

In [9]:
# ---------------------------------------------------
# 환경별 프로젝트 라우팅 패턴
# ---------------------------------------------------
# 실제 서비스에서 사용하는 환경별 Trace 라우팅이에요

def get_project_name(environment: str) -> str:
    """환경에 따라 LangSmith 프로젝트를 선택해요"""
    project_map = {
        "development": "MyApp-DEV",
        "staging": "MyApp-STAGING",
        "production": "MyApp-PROD"
    }
    return project_map.get(environment, "MyApp-DEV")

def handle_user_request(user_id: str, question: str, environment: str = "development") -> str:
    """사용자 요청을 처리하며 적절한 프로젝트에 Trace를 기록해요"""
    project = get_project_name(environment)

    with ls.tracing_context(
        project_name=project,
        tags=[f"env:{environment}"],
        metadata={
            "user_id": user_id,
            "environment": environment
        }
    ):
        return chain.invoke({"topic": question})

# 개발 환경 요청
r1 = handle_user_request("dev_user", "RAG 시스템", "development")
print(f"[DEV] 응답: {r1[:100]}")

# 운영 환경 요청
r2 = handle_user_request("prod_user_001", "벡터 검색", "production")
print(f"[PROD] 응답: {r2[:100]}")

print("\n각 환경의 Trace가 별도 프로젝트에 기록됐어요!")

[DEV] 응답: RAG 시스템은 Retrieval-Augmented Generation의 약자로, 정보 검색과 생성 모델을 결합하여 보다 정확하고 풍부한 응답을 생성하는 AI 기술입니다.
[PROD] 응답: 벡터 검색은 고차원 공간에서 데이터 포인트를 벡터로 표현하고, 이들 벡터 간의 유사성을 기반으로 관련 정보를 효율적으로 검색하는 방법입니다.

각 환경의 Trace가 별도 프로젝트에 기록됐어요!


## 6. LangSmith UI 필터링 가이드

지금까지 추가한 메타데이터와 태그를 LangSmith UI에서 필터링하는 방법을 배워요.

### 필터 옵션

| 필터 유형 | 사용 방법 | 예시 |
|-----------|-----------|------|
| Run Name | `run_name contains "rag"` | RAG 관련 Trace 검색 |
| Tags | `tags contains "production"` | 운영 환경 Trace만 |
| Metadata | `metadata.environment = "dev"` | 개발 환경 Trace만 |
| Status | `status = "error"` | 실패한 Trace만 |
| Token 수 | `total_tokens > 1000` | 토큰 많은 Trace |
| 시간 | 날짜 범위 선택 | 특정 기간 Trace |

> 🎯 **강의 포인트**: LangSmith UI에서 직접 필터를 적용해 보세요. 태그 `nlp`으로 필터링하거나 `metadata.category = "nlp_model"`로 특정 Trace를 찾아보세요.

## 7. 조건부 추적과 샘플링

모든 호출을 추적하면 비용이 증가해요. `tracing_context(enabled=False)`로 특정 호출을 제외하거나, 샘플링으로 비율을 조절할 수 있어요.

> ⚠️ **자주 하는 실수**: 민감한 개인정보(이름, 이메일, 전화번호)가 포함된 데이터는 추적에서 제외해야 해요. `tracing_context(enabled=False)`를 활용하세요.

In [10]:
# ---------------------------------------------------
# 조건부 추적: 민감 데이터 제외
# ---------------------------------------------------
# 특정 조건에서 추적을 비활성화해요

def safe_chat(user_input: str, is_sensitive: bool = False) -> str:
    """민감 데이터는 추적에서 제외하는 함수예요"""
    if is_sensitive:
        # 민감 데이터: 추적 비활성화
        with ls.tracing_context(enabled=False):
            return chain.invoke({"topic": user_input})
    else:
        # 일반 데이터: 정상 추적
        return chain.invoke({"topic": user_input})

# 일반 질문: 추적됨
result_public = safe_chat("머신러닝이란?")
print(f"일반 질문 (추적됨): {result_public[:100]}")

# 민감 데이터: 추적 안 됨
result_private = safe_chat("사용자 Alice의 개인 정보 요약", is_sensitive=True)
print(f"민감 데이터 (미추적): {result_private[:100]}")

print("\nLangSmith에서 일반 질문의 Trace만 보여요. 민감 데이터는 기록되지 않아요.")

일반 질문 (추적됨): 머신러닝은 데이터로부터 패턴을 학습하여 예측이나 결정을 자동으로 수행하는 인공지능의 한 분야입니다.
민감 데이터 (미추적): 사용자 Alice의 개인 정보 요약은 그녀의 기본 정보, 관심사, 활동 및 선호도를 포함하여 그녀를 이해하는 데 필요한 핵심적인 데이터를 제공합니다.

LangSmith에서 일반 질문의 Trace만 보여요. 민감 데이터는 기록되지 않아요.


In [11]:
# ---------------------------------------------------
# 샘플링: 전체 요청 중 일부만 추적
# ---------------------------------------------------
# 운영 환경에서 비용 절감을 위해 일부만 추적해요
import random

SAMPLING_RATE = 0.5  # 50%만 추적 (0.0 ~ 1.0)

def sampled_chat(question: str) -> str:
    """샘플링 비율에 따라 조건부로 추적해요"""
    should_trace = random.random() < SAMPLING_RATE

    with ls.tracing_context(
        enabled=should_trace,
        metadata={"sampled": should_trace, "sampling_rate": SAMPLING_RATE}
    ):
        return chain.invoke({"topic": question})

# 10번 호출 중 약 50%만 추적
print(f"샘플링 비율: {SAMPLING_RATE * 100:.0f}%")
print("10번 호출 중 약 절반만 LangSmith에 기록돼요")

topics = ["딥러닝", "NLP", "컴퓨터 비전", "강화학습", "벡터 DB",
        "임베딩", "파인튜닝", "RAG", "에이전트", "멀티모달"]

for t in topics[:5]:  # 5개만 테스트
    result = sampled_chat(t)
    print(f"  {t}: {result[:60]}...")

샘플링 비율: 50%
10번 호출 중 약 절반만 LangSmith에 기록돼요
  딥러닝: 딥러닝은 인공신경망을 기반으로 한 기계 학습의 한 분야로, 대량의 데이터를 통해 패턴을 학습하고 복잡한 문제...
  NLP: 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 해석하며 생성할 수 있도록 하는 인공지능의 한 분야입...
  컴퓨터 비전: 컴퓨터 비전은 컴퓨터가 이미지나 비디오에서 정보를 추출하고 이해할 수 있도록 하는 기술 및 연구 분야입니다....
  강화학습: 강화학습은 에이전트가 환경과 상호작용하며 보상을 최대화하기 위해 최적의 행동 전략을 학습하는 기계 학습의 한...
  벡터 DB: 벡터 DB는 고차원 벡터 데이터를 저장하고 검색할 수 있도록 최적화된 데이터베이스로, 주로 머신러닝 및 인공...


## 8. 실습: 메타데이터 기반 대시보드 구성

In [12]:
# ============================================================
# TODO: 아래 코드에서 metadata와 tags를 커스터마이징해서 실행하고
#       LangSmith UI에서 각 필터링 기능을 사용해 보세요
# 힌트: user_type, topic_category 등 다양한 메타데이터를 추가해 보세요
# 예상 결과: 각 메타데이터 키로 LangSmith에서 필터링이 가능해져요
# ============================================================

from langchain_core.tracers.langchain import wait_for_all_tracers

# 실습: 다양한 사용자와 카테고리 시뮬레이션
test_cases = [
    {"user_id": "u001", "topic": "LSTM", "category": "nlp", "tags": ["advanced"]},
    {"user_id": "u002", "topic": "CNN", "category": "cv", "tags": ["beginner"]},
    {"user_id": "u001", "topic": "어텐션 메커니즘", "category": "nlp", "tags": ["advanced"]},
    # TODO: 여기에 새로운 테스트 케이스를 추가해 보세요!
]

for case in test_cases:
    result = chain.invoke(
        {"topic": case["topic"]},
        config={
            "run_name": f"explain-{case['topic']}",
            "tags": case["tags"] + [f"category:{case['category']}"],  # 태그에 카테고리 추가
            "metadata": {
                "user_id": case["user_id"],
                "category": case["category"]
                # TODO: 여기에 더 많은 메타데이터를 추가해 보세요!
            }
        }
    )
    print(f"[{case['user_id']}] {case['topic']}: {result[:60]}...")

# 모든 Trace 전송 완료
wait_for_all_tracers()
print("\nLangSmith에서 다음을 시도해 보세요:")
print("  1. tags contains 'advanced' 필터로 고급 주제만 보기")
print("  2. metadata.user_id = 'u001' 필터로 특정 사용자 Trace 모으기")
print("  3. metadata.category = 'nlp' 필터로 NLP 주제만 보기")

[u001] LSTM: LSTM(장기 단기 기억)은 시퀀스 데이터의 장기 의존성을 학습하기 위해 설계된 순환 신경망(RNN) 구조로...
[u002] CNN: CNN(Convolutional Neural Network)은 주로 이미지 인식 및 처리에 사용되는 딥러닝 ...
[u001] 어텐션 메커니즘: 어텐션 메커니즘은 입력 데이터의 특정 부분에 집중하여 중요한 정보를 강조하고, 이를 통해 모델의 성능을 향상...

LangSmith에서 다음을 시도해 보세요:
  1. tags contains 'advanced' 필터로 고급 주제만 보기
  2. metadata.user_id = 'u001' 필터로 특정 사용자 Trace 모으기
  3. metadata.category = 'nlp' 필터로 NLP 주제만 보기


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **config 파라미터**: `run_name`, `tags`, `metadata`를 단일 호출에 적용하는 방법이에요
- **@traceable 메타데이터**: 함수 정의 시 정적 메타데이터를 설정해요
- **get_current_run_tree()**: 실행 중에 동적으로 메타데이터와 태그를 추가해요
- **tracing_context**: 새 Span 없이 블록 내 기본 메타데이터를 설정해요 (Span 생성 안 함)
- **프로젝트 분리**: dev/staging/prod 환경별로 별도 LangSmith 프로젝트를 사용해요
- **조건부 추적**: 민감 데이터 제외, 샘플링으로 추적 비용을 관리해요

## 다음 노트북 예고

다음 `07-Document-Loading-Tracing.ipynb`에서는 **RAG 파이프라인의 전 단계 추적**을 시작해요. 문서 로딩, 청킹, 임베딩, 벡터 저장소 구축까지 각 단계가 LangSmith에서 어떻게 기록되는지 살펴볼게요.